# Evaluation & Visualization of LLaVA Zero-Shot Facial Expression Classification

This notebook analyzes the results of **zero-shot facial expression classification** using LLaVA v1.6 (Mistral-7B) across **4 benchmark datasets**:

| Dataset | Images | Classes | Source |
|---------|--------|---------|--------|
| **FER2013** | 3,589 | 7 | Kaggle |
| **CK+** | 981 | 7 | Cohn-Kanade |
| **RAF-DB** | 3,068 | 7 | Real-world Affective Faces |
| **AffectNet** | 3,200 | 8 | Mollahosseini et al. |

The model was **not fine-tuned** on any of these datasets. All predictions were made using a single text prompt.

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully.')

## 2. Load & Parse Results

The raw classification results are stored in `LLAVA VERİLER.xlsx`, where each sheet contains a **confusion matrix** for one dataset.

**Sheet structure:**
- Row 0: True class names (column headers)
- Row 1: Total image count per class
- Rows 2+: Predicted class name + count per true class

In [ ]:
EXCEL_PATH = "../data/LLAVA VERİLER.xlsx"

def parse_confusion_sheet(xlsx_path, sheet_name):
    """Parse a sheet from the Excel file into a confusion matrix."""
    df = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)
    
    # Extract true class names from row 0 (every 3rd column)
    true_classes = []
    class_cols = []
    for col_idx in range(0, df.shape[1], 3):
        val = df.iloc[0, col_idx]
        if pd.notna(val):
            true_classes.append(str(val).strip())
            class_cols.append(col_idx + 1)
    
    # Extract total images per class from row 1
    totals = [int(df.iloc[1, ci]) for ci in class_cols]
    
    # Extract predicted class names from rows 2+
    pred_classes = []
    for row_idx in range(2, df.shape[0]):
        val = df.iloc[row_idx, 0]
        if pd.notna(val) and str(val).strip():
            pred_classes.append(str(val).strip())
        else:
            break
    
    # Build confusion matrix
    n_pred = len(pred_classes)
    n_true = len(true_classes)
    cm = np.zeros((n_pred, n_true), dtype=int)
    for j, ci in enumerate(class_cols):
        for i in range(n_pred):
            val = df.iloc[2 + i, ci]
            if pd.notna(val):
                cm[i, j] = int(val)
    
    return true_classes, pred_classes, cm, totals

# Parse all 4 datasets
datasets = {}
for sheet in ['fer2013', 'CK+', 'RAF-DB', 'AffectNet']:
    true_cls, pred_cls, cm, totals = parse_confusion_sheet(EXCEL_PATH, sheet)
    datasets[sheet] = {
        'true_classes': true_cls,
        'pred_classes': pred_cls,
        'confusion_matrix': cm,
        'totals': totals
    }
    acc = np.trace(cm) / cm.sum() * 100  # diagonal sum / total
    print(f'{sheet:12s} | {cm.sum():>5} images | {len(true_cls)} classes | Accuracy: {acc:.2f}%')

print('\nAll datasets loaded successfully.')

## 3. Confusion Matrix Heatmaps

A **confusion matrix** shows how often each predicted class matches each true class. The diagonal represents correct predictions — higher values on the diagonal mean better performance.

We normalize each column by the total number of images in that class to get **percentages**, making it easier to compare across classes with different sample sizes.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 16))
fig.suptitle('Confusion Matrices — LLaVA v1.6 Zero-Shot Classification', fontsize=16, fontweight='bold', y=1.02)

for idx, (name, data) in enumerate(datasets.items()):
    ax = axes[idx // 2][idx % 2]
    cm = data['confusion_matrix']
    true_cls = data['true_classes']
    pred_cls = data['pred_classes']
    
    # Normalize by column (true class totals) to get percentages
    col_sums = cm.sum(axis=0, keepdims=True)
    col_sums[col_sums == 0] = 1  # avoid division by zero
    cm_norm = cm / col_sums * 100
    
    # Clean class names for display
    true_labels = [c.replace('DİSGUST', 'Disgust').replace('SURPRİSED', 'Surprised')
                    .replace('SURPRİSE', 'Surprise').replace('CONTEMPT - Küçümseme', 'Contempt')
                    .title() for c in true_cls]
    pred_labels = [c.title() for c in pred_cls]
    
    acc = np.trace(cm) / cm.sum() * 100
    
    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues', ax=ax,
                xticklabels=true_labels, yticklabels=pred_labels,
                vmin=0, vmax=100, cbar_kws={'label': '%'})
    ax.set_title(f'{name} (Accuracy: {acc:.1f}%)', fontsize=13, fontweight='bold')
    ax.set_xlabel('True Label', fontsize=11)
    ax.set_ylabel('Predicted Label', fontsize=11)
    ax.tick_params(axis='both', labelsize=9)

plt.tight_layout()
plt.savefig('../assets/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to assets/confusion_matrices.png')

## 4. Per-Class Performance Metrics

For each dataset, we calculate:
- **Precision**: Of all images predicted as class X, how many were actually X?
- **Recall**: Of all images truly belonging to class X, how many were correctly predicted?
- **F1-Score**: Harmonic mean of precision and recall — a balanced metric

In [ ]:
def calculate_metrics(cm, true_classes, pred_classes):
    """Calculate per-class precision, recall, F1 from confusion matrix."""
    # Build y_true and y_pred from the confusion matrix
    y_true = []
    y_pred = []
    
    for j, true_cls in enumerate(true_classes):
        for i, pred_cls in enumerate(pred_classes):
            count = cm[i, j]
            for _ in range(count):
                y_true.append(true_cls.lower().replace('dİsgust','disgust')
                              .replace('süprİsed','surprised').replace('süprİse','surprise')
                              .replace('contempt - küçümseme','contempt').strip())
                y_pred.append(pred_cls.lower().strip())
    
    return y_true, y_pred

# Generate classification reports
all_reports = {}
for name, data in datasets.items():
    y_true, y_pred = calculate_metrics(
        data['confusion_matrix'], data['true_classes'], data['pred_classes']
    )
    
    # Get unique labels (union of true and pred)
    labels = sorted(set(y_true) | set(y_pred))
    
    report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    all_reports[name] = report
    
    print(f'\n{"="*60}')
    print(f'{name} — Classification Report')
    print(f'{"="*60}')
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

## 5. Per-Class Recall (Accuracy) Comparison

This chart shows how well the model recognizes **each emotion** across datasets. Recall = "out of all true X images, how many did the model correctly identify?"

In [ ]:
# Collect recall per class per dataset
# Use common emotion names
common_emotions = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprised', 'neutral']
emotion_aliases = {
    'anger': 'angry', 'sadness': 'sad', 'surprise': 'surprised',
    'surprised': 'surprised', 'neutral': 'neutral'
}

recall_data = {}
for name, report in all_reports.items():
    recall_data[name] = {}
    for cls, metrics in report.items():
        if isinstance(metrics, dict) and 'recall' in metrics:
            normalized = emotion_aliases.get(cls, cls)
            if normalized in common_emotions:
                recall_data[name][normalized] = metrics['recall'] * 100

# Create grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(common_emotions))
width = 0.18
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for i, (name, recalls) in enumerate(recall_data.items()):
    values = [recalls.get(e, 0) for e in common_emotions]
    bars = ax.bar(x + i * width, values, width, label=name, color=colors[i], alpha=0.85)
    # Add value labels on bars
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                    f'{val:.0f}%', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xlabel('Emotion Class', fontsize=12)
ax.set_ylabel('Recall (%)', fontsize=12)
ax.set_title('Per-Class Recall Across Datasets — LLaVA v1.6 Zero-Shot', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([e.capitalize() for e in common_emotions], fontsize=11)
ax.set_ylim(0, 110)
ax.legend(fontsize=10, loc='upper right')
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.3, label='50% baseline')

plt.tight_layout()
plt.savefig('../assets/per_class_recall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to assets/per_class_recall.png')

## 6. Overall Dataset Comparison

A summary comparing the model's overall performance across all 4 datasets.

In [ ]:
# Summary table
summary_rows = []
for name, data in datasets.items():
    cm = data['confusion_matrix']
    total = cm.sum()
    correct = np.trace(cm)
    acc = correct / total * 100
    
    report = all_reports[name]
    macro_f1 = report['macro avg']['f1-score'] * 100
    weighted_f1 = report['weighted avg']['f1-score'] * 100
    
    summary_rows.append({
        'Dataset': name,
        'Images': total,
        'Classes': len(data['true_classes']),
        'Accuracy (%)': round(acc, 2),
        'Macro F1 (%)': round(macro_f1, 2),
        'Weighted F1 (%)': round(weighted_f1, 2)
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print()

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(summary_rows))
width = 0.25

bars1 = ax.bar(x - width, [r['Accuracy (%)'] for r in summary_rows], width, label='Accuracy', color='#2196F3')
bars2 = ax.bar(x, [r['Macro F1 (%)'] for r in summary_rows], width, label='Macro F1', color='#4CAF50')
bars3 = ax.bar(x + width, [r['Weighted F1 (%)'] for r in summary_rows], width, label='Weighted F1', color='#FF9800')

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Dataset', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Overall Performance Comparison — LLaVA v1.6 Zero-Shot', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([r['Dataset'] for r in summary_rows], fontsize=11)
ax.set_ylim(0, 80)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../assets/overall_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to assets/overall_comparison.png')

## 7. Key Findings & Analysis

### Performance Summary

| Rank | Dataset | Accuracy | Macro F1 | Observation |
|------|---------|----------|----------|-------------|
| 1 | **RAF-DB** | ~62% | Best | Real-world images, model handles natural expressions well |
| 2 | **FER2013** | ~43% | Moderate | Low-res 48x48 grayscale images hurt vision encoder |
| 3 | **CK+** | ~28% | Low | Lab-posed expressions differ from model's training data |
| 4 | **AffectNet** | ~25% | Lowest | 8 classes (contempt added), high inter-class confusion |

### Emotion-Level Insights

- **Happy** is consistently the **best recognized** emotion across all datasets — its facial features (smile, raised cheeks) are highly distinctive
- **Fear** and **Disgust** are the **hardest** to classify — the model frequently confuses them with Surprise and Angry
- **Surprise** tends to be **over-predicted** — the model defaults to it when uncertain
- **Contempt** (where applicable) is almost never correctly predicted — this subtle expression is challenging even for supervised models

### Why These Results Matter

Zero-shot performance of ~25–62% may seem low compared to supervised models (which typically achieve 70-90%). However:

1. **No training data was used** — the model generalizes purely from pre-trained knowledge
2. **Single prompt** — no prompt engineering optimization was performed
3. **Image quality varies dramatically** — from 48x48 grayscale (FER2013) to high-res photos (RAF-DB)
4. **RAF-DB's 62%** is competitive with some early supervised approaches on the same dataset

### Limitations

- Prompt sensitivity: different wordings may yield different results
- The model tends to produce verbose outputs that need post-processing
- Low-resolution images significantly degrade performance
- Class imbalance in datasets affects weighted metrics

### Future Directions

- Fine-tune LLaVA using LoRA/QLoRA on these datasets
- Experiment with prompt engineering (chain-of-thought, few-shot examples)
- Compare with other multimodal models (GPT-4V, Gemini, InternVL)
- Evaluate on video-based emotion recognition datasets